In [1]:
import os

from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain_core.messages import BaseMessage
from langchain_openai.chat_models.base import BaseChatOpenAI


class ChildChatOpenAI(BaseChatOpenAI):
    def get_num_tokens_from_messages(self, messages: list[BaseMessage]) -> int:
        model, encoding = self._get_encoding_model()
        # 注意：这里只是解决了计算的问题，具体计算方案具体准确取决于具体模型
        if model.startswith("qwen-turbo"):
            # 调用祖父类的函数
            return super(BaseChatOpenAI, self).get_num_tokens_from_messages(messages)
        else:
            return super().get_num_tokens_from_messages(messages)

In [3]:
llm = ChildChatOpenAI(
    model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),  # 若走原生 OpenAI，可传 None/不传
    temperature=0.2,  # 稳定输出
    timeout=1200,  # 超时保护（秒）
    max_retries=2,  # 简单重试
)

In [4]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.messages import SystemMessage

# 1. 初始化一个空的记忆历史对象
history = InMemoryChatMessageHistory()

# 2. 添加一条系统消息，为AI设定角色
history.add_message(SystemMessage(content="你是一个有用的AI助手，请用中文回答问题。"))
print("✓ 添加系统消息")

# 3. 使用便捷方法添加用户消息
history.add_user_message("你好，我是小明")
print("✓ 添加用户消息: 你好，我是小明")

# 4. 使用便捷方法添加AI消息
history.add_ai_message("你好小明！很高兴认识你。有什么我可以帮助你的吗？")
print("✓ 添加AI消息: 你好小明！很高兴认识你。...")

# 5. 查看当前存储的所有消息
print("\n--- 当前消息历史 ---")
print(history.messages)

✓ 添加系统消息
✓ 添加用户消息: 你好，我是小明
✓ 添加AI消息: 你好小明！很高兴认识你。...

--- 当前消息历史 ---
[SystemMessage(content='你是一个有用的AI助手，请用中文回答问题。', additional_kwargs={}, response_metadata={}), HumanMessage(content='你好，我是小明', additional_kwargs={}, response_metadata={}), AIMessage(content='你好小明！很高兴认识你。有什么我可以帮助你的吗？', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]


In [5]:
# --- 准备工作 ---
history = InMemoryChatMessageHistory()
history.add_message(
    SystemMessage(
        content="你是一个有用的AI助手。请记住用户告诉你的信息，并在后续对话中使用。"
    )
)

# --- 第一轮对话：用户提供信息 ---
user_input1 = "我叫张三，今年25岁，是一名软件工程师"
history.add_user_message(user_input1)

# 调用LLM，传入完整历史
response1 = llm.invoke(history.messages)
ai_response1 = response1.content
history.add_ai_message(ai_response1)

print(f"用户: {user_input1}")
print(f"AI: {ai_response1}\n")

# --- 第二轮对话：测试记忆 ---
user_input2 = "我的职业是什么？"
history.add_user_message(user_input2)

# 再次调用LLM，传入【更新后】的完整历史
response2 = llm.invoke(history.messages)
ai_response2 = response2.content
history.add_ai_message(ai_response2)

print(f"用户: {user_input2}")
print(f"AI: {ai_response2}\n")

# --- 查看最终的对话历史 ---
print("--- 最终消息历史 ---")
for msg in history.messages:
    print(f"[{type(msg).__name__}] {msg.content}")

用户: 我叫张三，今年25岁，是一名软件工程师
AI: <think>
好的，用户告诉我他叫张三，25岁，是软件工程师。我需要记住这些信息，并在后续对话中使用。首先，我应该确认这些信息是否正确，并表达欢迎。然后，可能需要询问他是否有具体的问题或需要帮助的地方，比如技术问题、职业发展建议，或者项目上的帮助。作为软件工程师，他可能对编程语言、工具使用、代码优化、职业规划等方面感兴趣。我需要保持友好和专业的语气，同时准备好根据他的需求提供具体帮助。另外，要注意他可能没有明确说明的需求，比如学习资源推荐或者行业动态。需要确保回应简洁，不使用Markdown格式，保持自然的口语化中文。
</think>

你好，张三！很高兴认识你。作为一名25岁的软件工程师，你正处于职业发展的黄金阶段。最近在忙什么有趣的项目吗？或者有什么技术难题需要帮忙解答的？

用户: 我的职业是什么？
AI: <think>
好的，用户问“我的职业是什么？”，我需要先回顾之前的对话。用户之前已经告诉过我他叫张三，25岁，是软件工程师。所以现在他问这个问题，可能是在确认我是否记住了他的信息，或者他可能想进一步讨论职业相关的话题。

首先，我要确认用户的需求。他可能只是想确认我是否记得他的职业，或者他可能有更深层次的问题，比如对职业发展的疑问，或者想分享最近的工作经历。考虑到之前的对话中用户提到过软件工程师，我应该先直接回答他的职业，然后提供进一步帮助的可能性。

接下来，我需要检查是否有其他可能的意图。比如，用户可能在测试我的记忆能力，或者他可能想转换话题到职业规划、技术问题等。因此，在回答时，除了重复他的职业信息，最好加上一个开放性的问题，鼓励他进一步交流。

另外，要注意保持回答的友好和自然，避免机械化的回复。使用用户的名字会让对话更亲切，比如“张三”再次被提及，可以增强互动感。同时，提到“职业发展的黄金阶段”可以呼应之前的对话内容，显示我确实记住了之前的交流。

最后，确保回答简洁明了，同时提供帮助的可能性，让用户感到被支持。如果有任何潜在的问题，比如用户可能对职业发展有困惑，或者需要技术建议，我的回答应该为后续对话留下空间。
</think>

你的职业是软件工程师，张三。这是一份充满挑战和创造力的工作，尤其是在你这个年龄段，职业发展机会非常多。需要聊聊技术方向、职业规划，还是最近的项目进展？



In [6]:
history = InMemoryChatMessageHistory()
history.add_message(SystemMessage(content="你是一个有用的助手"))
max_messages = 6  # 设置一个较小的最大消息数（包括系统消息）

for i in range(5):
    history.add_user_message(f"用户消息 {i+1}")
    history.add_ai_message(f"AI回复 {i+1}")

    # 检查并限制消息数量
    if len(history.messages) > max_messages:
        print(
            f"  消息数达到 {len(history.messages)}，超过限制 {max_messages}，开始截断..."
        )
        # 保留系统消息和最近的消息
        system_messages = [
            msg for msg in history.messages if isinstance(msg, SystemMessage)
        ]
        dialogue_messages = [
            msg for msg in history.messages if not isinstance(msg, SystemMessage)
        ]

        # 保留最近的 (max_messages - system_messages数量) 条对话消息
        history.messages = (
            system_messages
            + dialogue_messages[-(max_messages - len(system_messages)) :]
        )
        print(f"  截断后消息数为: {len(history.messages)}")

print("\n--- 最终消息状态 ---")
for msg in history.messages:
    print(f"[{type(msg).__name__.replace('Message','')}] {msg.content}")

  消息数达到 7，超过限制 6，开始截断...
  截断后消息数为: 6
  消息数达到 8，超过限制 6，开始截断...
  截断后消息数为: 6
  消息数达到 8，超过限制 6，开始截断...
  截断后消息数为: 6

--- 最终消息状态 ---
[System] 你是一个有用的助手
[AI] AI回复 3
[Human] 用户消息 4
[AI] AI回复 4
[Human] 用户消息 5
[AI] AI回复 5


In [7]:
def simple_token_count(messages):
    # 更准确的中文token计数方法：将每个中文字符视为一个token
    # 注意：这仍然是简化方法，实际LLM的tokenization更复杂
    return sum(len(msg.content) for msg in messages)


history = InMemoryChatMessageHistory()
max_tokens = 50  # 设置一个较小的最大Token数

messages_to_add = [
    ("user", "你好，我想了解人工智能的发展历史"),
    (
        "ai",
        "人工智能起源于20世纪50年代，当时科学家们开始探索让机器模拟人类智能的可能性",
    ),
    ("user", "那么机器学习和深度学习是什么时候发展起来的？"),
    ("ai", "机器学习在20世纪80年代开始兴起，而深度学习则在2010年代迎来了突破性发展"),
]

for msg_type, content in messages_to_add:
    if msg_type == "user":
        history.add_user_message(content)
    else:
        history.add_ai_message(content)

    token_count = simple_token_count(history.messages)
    print(f"添加消息后: Token数 = {token_count}")

    while token_count > max_tokens and len(history.messages) > 1:
        removed_msg = history.messages.pop(0)  # 移除最旧的消息
        token_count = simple_token_count(history.messages)
        print(f"  Token超限，移除一条旧消息。当前Token数: {token_count}")

print(
    "\n--- 最终消息状态 (Token数: {}) ---".format(simple_token_count(history.messages))
)
for msg in history.messages:
    print(f"[{type(msg).__name__.replace('Message','')}] {msg.content}")

添加消息后: Token数 = 16
添加消息后: Token数 = 55
  Token超限，移除一条旧消息。当前Token数: 39
添加消息后: Token数 = 61
  Token超限，移除一条旧消息。当前Token数: 22
添加消息后: Token数 = 61
  Token超限，移除一条旧消息。当前Token数: 39

--- 最终消息状态 (Token数: 39) ---
[AI] 机器学习在20世纪80年代开始兴起，而深度学习则在2010年代迎来了突破性发展


In [8]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

# 1. 定义消息历史存储（全局或按 Session 存储）
store: dict[str, InMemoryChatMessageHistory] = {}


def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]


# 2. 定义提示词模板
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "你是一个有用的助手。"),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{input}"),
    ]
)

# 3. 核心：实现“窗口记忆” (k=2 相当于保留最近 4 条消息：2人+2机)
# 使用 trim_messages 工具可以精确控制保留多少历史
chain = prompt | llm

# 4. 包装成带历史记录的链
runnable_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

# --- 模拟对话 ---
config = {"configurable": {"session_id": "user_123"}}

# 第一轮
print(
    "R1:",
    runnable_with_history.invoke({"input": "我叫李四，住在上海。"}, config).content,
)

# 第二轮
print("R2:", runnable_with_history.invoke({"input": "我喜欢编程。"}, config).content)


# 关键：手动修剪历史以模拟 WindowMemory (保持最近 k*2 条)
# 在实际生产中，通常在 prompt 链里加入 trim_messages 逻辑
def update_recent_history(history: InMemoryChatMessageHistory):
    # 模拟 k=1 (保留最近一轮对话，即 2 条消息)
    if len(history.messages) > 2:
        history.messages = history.messages[-2:]


# 第三轮：问名字
update_recent_history(get_session_history("user_123"))
print(
    "R3 (问名字):",
    runnable_with_history.invoke({"input": "我叫什么名字？"}, config).content,
)

# 第四轮：问住址
update_recent_history(get_session_history("user_123"))
print(
    "R4 (问住址):",
    runnable_with_history.invoke({"input": "我住在哪里？"}, config).content,
)

R1: <think>
好的，用户告诉我他叫李四，住在上海。首先，我需要确认用户的基本信息是否正确，但可能不需要直接回应这一点，因为用户可能只是自我介绍。接下来，我应该考虑用户可能的意图。他可能希望我记住他的名字和居住地，以便后续对话中使用。也有可能他想讨论与上海相关的话题，比如天气、生活、旅游景点或者工作机会等。

我需要保持友好和开放的态度，鼓励用户进一步说明需求。例如，可以询问他是否需要关于上海的建议，或者是否有其他问题需要帮助。同时，要注意不要假设用户的具体需求，而是提供广泛的支持选项。另外，考虑到用户可能来自不同的背景，可能需要用简单明了的中文交流，避免使用过于复杂的术语或地方方言，除非用户主动提及。

还要检查是否有潜在的信息需要进一步确认，比如用户是否希望长期对话，或者是否有特定的服务需求。例如，用户可能需要推荐上海的餐厅、交通信息，或者了解当地的文化活动。此外，考虑到上海是一个大城市，用户可能对生活成本、租房建议或旅游路线感兴趣。

最后，确保回应用户时使用他的名字，以增加亲切感，比如“李四，很高兴认识你！上海是个很棒的城市，有什么我可以帮你的吗？”这样既确认了信息，又开启了进一步对话的可能性。
</think>

李四，很高兴认识你！上海是个充满活力的城市，有什么我可以帮你的吗？比如推荐附近的美食、景点，或者需要了解天气、交通信息？随时告诉我哦！
R2: <think>
好的，用户李四告诉我他喜欢编程，我需要根据这个信息来调整回复。首先，用户之前提到他住在上海，现在又补充了兴趣是编程，这可能意味着他可能对技术社区、编程学习资源或者相关活动感兴趣。

我需要先确认用户的具体需求。他可能只是想分享兴趣，或者希望得到相关的建议或资源。比如，他可能想了解上海的编程活动、学习平台、或者职业发展建议。也有可能他需要帮助解决编程中的具体问题，或者寻找项目合作机会。

接下来，我应该提供一些具体的建议，涵盖学习资源、社区活动、职业发展等方面。例如，推荐在线学习平台如Coursera、LeetCode，或者上海本地的编程社区和Meetup活动。同时，可以询问他具体喜欢哪种编程语言或领域，以便更精准地推荐。

另外，考虑到用户可能希望进一步讨论，我需要保持开放式的提问，鼓励他分享更多细节，比如他目前的编程水平、感兴趣的领域（如Web开发、数据科学、人工智能等），或者是

In [9]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.messages import HumanMessage, AIMessage

# 1. 定义 Prompt 模板
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "你是一个乐于助人的助手。"),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{input}"),
    ]
)

# 2. 构建 LCEL 链
chain = prompt | llm

# 3. 管理历史记录 (替代旧版的 Memory 对象)
# 在新版中，通常通过一个字典来存储不同 session 的历史
store: dict[str, ChatMessageHistory] = {}


def get_session_history(session_id: str) -> ChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]


# 4. 使用 RunnableWithMessageHistory 包装
chain = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

# --- 模拟添加之前的对话历史 ---
session_id = "user_001"
history = get_session_history(session_id)

history.add_user_message("我喜欢古典音乐，特别是巴赫的作品。")
history.add_ai_message("巴赫确实是一位伟大的作曲家。")
history.add_user_message("我还喜欢在周末去爬山。")
history.add_ai_message("听起来很健康！")
history.add_user_message("上周末我去了黄山，风景很美。")
history.add_ai_message("黄山以其奇松、怪石、云海而闻名。")
history.add_user_message("是的，我还看到了日出。")
history.add_ai_message("那一定是一次难忘的经历。")

# 5. 测试对话
config = {"configurable": {"session_id": session_id}}
response: AIMessage = chain.invoke(
    {"input": "你能总结一下我们刚才的对话吗？"}, config=config
)

print("\nAI的回应:", response.content)


AI的回应: <think>
好的，用户让我总结一下刚才的对话。首先，我需要回顾之前的对话内容。用户一开始提到喜欢古典音乐，特别是巴赫的作品，我回应了对巴赫的赞美。接着用户说喜欢周末爬山，我回应说听起来健康。然后用户提到上周末去了黄山，风景很美，我补充了黄山的特色。用户确认并提到看到了日出，我表示那是一次难忘的经历。最后用户请求总结对话。

现在需要将这些内容连贯地总结出来，确保涵盖所有要点。用户可能希望得到一个简洁的回顾，确认对话内容是否正确，或者作为记录保存。用户可能没有明确说明，但可能希望确认我是否准确理解了他们的兴趣和经历，或者想分享给他人。需要保持自然，用中文口语化表达，避免使用格式或术语。同时要检查是否有遗漏的信息，比如黄山的奇松、怪石、云海，以及日出的部分。确保总结准确，没有错误，并且流畅易懂。
</think>

当然可以！我们刚才的对话主要围绕你的兴趣和经历展开：  
1. 你提到喜欢古典音乐，尤其是巴赫的作品。  
2. 你分享了周末爬山的习惯，并提到上周末去了黄山，欣赏了那里的奇松、怪石和云海。  
3. 你还特别提到在黄山看到了日出，认为这是一次难忘的经历。  
整体上，你通过音乐和自然景观表达了对艺术与自然之美的热爱。需要补充或调整的地方吗？ 😊


In [10]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.messages import HumanMessage, AIMessage

# 1. 模拟实体数据库 (实际应用中可对接 Redis 或 图数据库)
# 这里的逻辑是：AI 发现对话中提到这些 Key，就会把 Value 喂给模型
entity_knowledge_base = {
    "小王": "小王在一家叫 'InnovateTech' 的公司工作，职业是软件工程师。",
    "InnovateTech": "InnovateTech 是一家具备 50 人规模的人工智能初创公司，总部在上海。",
}


# 2. 定义辅助函数：提取上下文中的实体信息
def fetch_entity_info(input_data: dict) -> str:
    user_input = input_data["input"]
    found_entities = []
    for entity, info in entity_knowledge_base.items():
        if entity in user_input:
            found_entities.append(f"[{entity}]: {info}")

    return (
        "\n".join(found_entities) if found_entities else "当前输入中未匹配到已知实体。"
    )


# 3. 构建 Chat Prompt 模板
# 最新接口推荐区分 System, Placeholder (History) 和 Human
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是一个聪明的助手。以下是你掌握的背景实体信息：\n{entities}\n请结合这些信息和历史对话回答问题。",
        ),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{input}"),
    ]
)

# 4. 构建 LCEL 链 (The Chain)
# 使用 RunnablePassthrough 组织输入数据流
chain = (
    RunnablePassthrough.assign(entities=fetch_entity_info)
    | prompt
    | llm
    | StrOutputParser()
)

# 65. 管理对话历史
demo_history = ChatMessageHistory()


def chat_with_entity_memory(user_text: str):
    # 构建输入字典
    input_payload = {"input": user_text, "history": demo_history.messages}

    # 运行链
    response_text = chain.invoke(input_payload)

    # 更新持久化历史记录
    demo_history.add_user_message(user_text)
    demo_history.add_ai_message(response_text)

    return response_text


# --- 模拟对话流程 ---

print("=== 启动对话 (LCEL 模式) ===")

# 第一轮：提供信息
print(f"User: 我的朋友小王在一家叫'InnovateTech'的公司工作。")
print(
    f"AI: {chat_with_entity_memory('我的朋友小王在一家叫\'InnovateTech\'的公司工作。')}\n"
)

# 第二轮：基于实体的追问
print(f"User: 小王的公司是做什么的？")
print(f"AI: {chat_with_entity_memory('小王的公司是做什么的？')}\n")

# 第三轮：直接询问实体细节
print(f"User: InnovateTech 规模有多大？")
print(f"AI: {chat_with_entity_memory('InnovateTech 规模有多大？')}\n")

# 查看最终的历史记录
print("=== 最终对话历史 ===")
for i, msg in enumerate(demo_history.messages):
    role = "用户" if isinstance(msg, HumanMessage) else "助手"
    print(f"{i+1}. {role}: {msg.content}")

=== 启动对话 (LCEL 模式) ===
User: 我的朋友小王在一家叫'InnovateTech'的公司工作。
AI: <think>
好的，用户提到他的朋友小王在InnovateTech工作，而我已经知道小王是那里的软件工程师，公司规模50人，总部在上海。用户可能想确认信息或者询问更多关于小王或公司的细节。我需要先确认已知信息，然后看看是否需要补充其他可能的细节，比如公司业务领域（人工智能），或者询问用户是否有具体问题。同时要注意用户可能想了解小王的工作内容、公司环境，或者是否有合作机会。需要保持友好，提供有用的信息，并引导用户提出更具体的问题。
</think>

是的，小王目前在InnovateTech担任软件工程师。这家公司专注于人工智能领域，团队规模约50人，总部位于上海。如果您想了解更多关于小王的工作内容或公司业务，可以具体问我哦！

User: 小王的公司是做什么的？
AI: <think>
好的，用户问小王的公司是做什么的。根据之前的对话，用户已经知道小王在InnovateTech工作，而我之前提到过这家公司专注于人工智能领域。现在需要详细说明公司的业务范围。

首先，我需要确认之前的信息是否正确。用户的朋友小王确实在InnovateTech，而我的回答中提到公司专注于人工智能，团队规模50人，总部在上海。用户现在想知道公司具体做什么，所以需要更详细的业务描述。

接下来，我应该扩展之前的信息，补充具体的业务领域，比如机器学习、自然语言处理、计算机视觉等。同时，可以提到他们开发的产品或解决方案，比如智能客服系统、数据分析工具，或者为其他企业提供AI集成服务。此外，可能涉及的行业应用，如金融、医疗、零售等，这样用户能更清楚公司的应用场景。

还要注意用户可能的潜在需求。用户可能想知道小王的工作内容是否与这些业务相关，或者公司是否有发展前景。因此，在回答时可以适当提到公司的技术优势或市场地位，比如是否有知名客户或成功案例，这样能增加信息的可信度和深度。

另外，需要保持回答的口语化和简洁，避免使用过于专业的术语，让用户容易理解。同时，确保信息准确，不添加未提及的内容，比如如果公司没有公开的具体产品，就不应该编造。但根据之前的设定，可以合理扩展已知的信息。

最后，检查是否有遗漏或需要补充的部分，比如公司的成立时间、创始人背景等，但如果没有相关信息

In [11]:
from pathlib import Path
from redislite import Redis
from langchain_community.chat_message_histories import RedisChatMessageHistory

# 1. 初始化 redislite
# 这会在当前目录下创建一个名为 'langchain_history.db' 的文件，类似于 SQLite
# 如果文件已存在，它会加载旧数据，实现真正的持久化
redis_file = Path("data") / "langchain_history.db"
native_redis = Redis(redis_file)
redis_url = f"unix://localhost:{native_redis.socket_file}"

# 2. 定义 session_id
session_id_user_A = "user_session_A_123"
session_id_user_B = "user_session_B_456"

# 3. 初始化历史记录对象
history_A = RedisChatMessageHistory(session_id=session_id_user_A, url=redis_url)
history_B = RedisChatMessageHistory(session_id=session_id_user_B, url=redis_url)

# 4. 添加内容
history_A.add_user_message("我喜欢喝咖啡")
history_A.add_ai_message("知道了，你喜欢喝咖啡。")

history_B.add_user_message("我是个程序员")
history_B.add_ai_message("很棒！编程是一个很有趣的职业。")

# 5. 验证隔离性
print(f"会话A的历史: {[msg.content for msg in history_A.messages]}")
print(f"会话B的历史: {[msg.content for msg in history_B.messages]}")

# 6. 模拟应用重启（重新获取 client 并读取）
print("\n--- 模拟应用重启 ---")
# 再次连接同一个 db 文件
reloaded_native_redis = Redis(redis_file)
reloaded_redis_url = f"unix://localhost:{reloaded_native_redis.socket_file}"

reloaded_history_A = RedisChatMessageHistory(
    session_id=session_id_user_A, url=reloaded_redis_url
)
print(f"重新加载的会话A历史: {[msg.content for msg in reloaded_history_A.messages]}")

会话A的历史: ['我喜欢喝咖啡', '知道了，你喜欢喝咖啡。']
会话B的历史: ['我是个程序员', '很棒！编程是一个很有趣的职业。']

--- 模拟应用重启 ---
重新加载的会话A历史: ['我喜欢喝咖啡', '知道了，你喜欢喝咖啡。']


In [12]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory

# 1. 定义一个包含历史记录占位符的LCEL链
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant."),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{input}"),
    ]
)
chain = prompt | llm

# 2. 准备一个存储，用于根据session_id管理多个history对象
store: dict[str, ChatMessageHistory] = {}


def get_session_history(session_id: str) -> ChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]


# 3. 使用RunnableWithMessageHistory包装链
chain = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

# 4. 调用时，传入config指定session_id
config = {"configurable": {"session_id": "my_session_1"}}

# --- 模拟对话 ---
response1: AIMessage = chain.invoke({"input": "Hi! I'm Bob"}, config=config)
print(response1)

response2: AIMessage = chain.invoke({"input": "What's my name?"}, config=config)
print(response2)


# 5. 检查底层存储
print("--- 底层存储内容 ---")
print(store["my_session_1"].messages)

content='<think>\nOkay, the user introduced himself as Bob. I should respond in a friendly and welcoming manner. Let me make sure to acknowledge his introduction and offer assistance. I\'ll keep it simple and open-ended to encourage him to ask questions or share what\'s on his mind. Something like, "Hi Bob! How can I assist you today?" That should work.\n</think>\n\nHi Bob! How can I assist you today?' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 86, 'prompt_tokens': 24, 'total_tokens': 110, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'Qwen/Qwen3-32B', 'system_fingerprint': None, 'id': 'chatcmpl-e73e693bfe1042fda7a36d2596f320aa', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019dc465-a850-7800-bdad-3d0277dd58e2-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 24, 'output_tokens': 86, 'total_tokens': 110, 'input_token_details': {}, 'output_token_